In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers numpy torch

In [3]:
import torch
import joblib
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./drive/MyDrive/l2_models/Лояльность_и_продажи"
MAX_LEN = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
mlb = joblib.load(f"{MODEL_PATH}/level2_mlb.joblib")
model.eval()

def predict_level2_above_thresh(text, threshold=0.3):
    """
    Возвращает все Level 2 категории с вероятностью выше threshold.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(0)
        probs = torch.sigmoid(logits).cpu().numpy()

    idx_above = np.where(probs > threshold)[0]
    if len(idx_above) == 0:
        idx_above = [np.argmax(probs)]
    labels = np.array(mlb.classes_)[idx_above]
    scores = probs[idx_above]

    result = [(label, round(float(score), 4)) for label, score in zip(labels, scores)]
    result = sorted(result, key=lambda x: x[1], reverse=True)
    return result

if __name__ == "__main__":
    test_texts = [
        "Привет! Подскажите, как узнать сколько миль у меня осталось на карте S7 Priority? Хочу спланировать покупку билета на ближайший вылет.",
        "Здравствуйте! Обращаюсь по поводу акции \"Летим в Сочи\". Не могу понять, какие рейсы участвуют? И как применить промокод при оплате? Спасибо.",
        "Добрый день! У меня сгорело 5000 миль! Почему мне не пришло уведомление? Это просто ужас, я их копил полгода! Требую восстановить.",
        "Скажите, а есть какие-то специальные тарифы для студентов? Летаю часто по учебе, очень нужна скидка. Спасибо за ответ!",
        "Пытаюсь оплатить часть билета милями, но система выдает ошибку. Пишет \"Недостаточно миль\", хотя у меня 15000. Что делать?",
        "Вижу что у вас идет распродажа \"Осенние каникулы\", но цены все равно выше чем у конкурентов. Будет ли еще снижение цен?",
        "Здраствуйте! У меня вопросс по начислению миль за последний перелет Москва-Казань. Должны были начислить 500 миль, но их нету.",
        "Подскажите, в чем разница между тарифами \"Эконом\" и \"Эконом гибкий\"? Хочу понять, стоит ли переплачивать за возможность изменений.",
        "Можно ли как-то ускорить начисление миль? Летел вчера, а мили до сих пор не пришли. Обычно быстрее приходили.",
        "Почему у вас такие дорогие тарифы на бизнес-класс? У Аэрофлота в 1.5 раза дешевле на те же даты. Есть ли какие-то акции?",
        "Хочу передать часть миль сыну для покупки билета. Как это сделать и есть ли комиссия? Спасибо!",
        "Заметила что цены на билеты вечером ниже чем утром. Это какая-то акция или просто совпадение? Когда лучше покупать?",
        "Не приходят уведомления о состоянии счета миль. Проверьте пожалуйста настройки и пришлите историю начислений за последние 3 месяца.",
        "Есть ли у вас программа рассрочки на дорогие билеты? Хочу лететь в отпуск, но сразу всю сумму платить тяжело.",
        "Почему при оплате билета милями списывается больше чем показывалось в калькуляторе? Это обман!",
        "Вижу что у вас есть \"Тариф для своих\", но не понимаю как на него попасть. Нужно быть определенное количество раз полететь?",
        "Как восстановить карту S7 Priority если я ее потерял? И не смогут ли ей воспользоваться другие люди?",
        "Почему на один и тот же рейс такие разные цены в разные дни? Это манипулирование ценами! Хочу понять логику ценообразования.",
        "Можно ли копить мили вместе с семьей? У нас с мужем отдельные карты, но хотим объединить мили для общего путешествия.",
        "Будет ли черная пятница у S7? Жду больших скидок на билеты зарубеж, очень хочется съездить в отпуск.",
        "Почему мне не начисляются мили за перелеты партнерами? Летал Turkish Airlines по коду S7, мили не пришли.",
        "Чем отличается \"бизнес-гибкий\" от обычного бизнеса? Стоит ли переплачивать если даты точно известны?",
        "Мой статус в программе лояльности понизился без предупреждения! Почему? Я регулярно летаю вашими авиалиниями.",
        "Есть ли скрытые платежи в ваших тарифах? При выборе билета одна цена, а при оплате оказывается на 20% дороже.",
        "Как узнать когда истекает срок действия моих миль? Хочу спланировать их использование чтобы ничего не сгорело.",
        "Почему у вас нет семейных тарифов? Летим вчетвером, а покупать отдельные билеты невыгодно. Есть ли групповые скидки?",
        "Не могу войти в аккаунт S7 Priority, пишет неверный пароль. Восстановление не работает, присылает ошибку. Помогите!",
        "Заметила что цены растут если часто ищешь один и тот же рейс. Это специальная система или просто кажется?",
        "Можно ли оплатить милями полностью весь билет или только часть? И есть ли ограничения по направлениям?"
    ]

    for i, text in enumerate(test_texts):
        prediction = predict_level2_above_thresh(text, threshold=0.3)
        print(f"\nТекст {i+1}: {text}")
        print("Категории с вероятностью > 0.3:")
        for label, prob in prediction:
            print(f"  - {label}: {prob}")



Текст 1: Привет! Подскажите, как узнать сколько миль у меня осталось на карте S7 Priority? Хочу спланировать покупку билета на ближайший вылет.
Категории с вероятностью > 0.3:
  - Программа лояльности: 0.9901

Текст 2: Здравствуйте! Обращаюсь по поводу акции "Летим в Сочи". Не могу понять, какие рейсы участвуют? И как применить промокод при оплате? Спасибо.
Категории с вероятностью > 0.3:
  - Акции тарифы: 0.9931

Текст 3: Добрый день! У меня сгорело 5000 миль! Почему мне не пришло уведомление? Это просто ужас, я их копил полгода! Требую восстановить.
Категории с вероятностью > 0.3:
  - Программа лояльности: 0.9903

Текст 4: Скажите, а есть какие-то специальные тарифы для студентов? Летаю часто по учебе, очень нужна скидка. Спасибо за ответ!
Категории с вероятностью > 0.3:
  - Акции тарифы: 0.9941

Текст 5: Пытаюсь оплатить часть билета милями, но система выдает ошибку. Пишет "Недостаточно миль", хотя у меня 15000. Что делать?
Категории с вероятностью > 0.3:
  - Программа лояльности: 

In [5]:
MODEL_PATH = "./drive/MyDrive/l2_models/После_вылета"
MAX_LEN = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
mlb = joblib.load(f"{MODEL_PATH}/level2_mlb.joblib")
model.eval()

test_texts_after = [
    "Требую вернуть деньги за билет! Рейс был отменен, а альтернативный вариант мне не подошел. Как быстро происходит возврат средств на карту? Это просто невыносимо!",
    "Хочу перенести свой вылет на следующую неделю. Как это сделать? У меня тариф 'Гибкий'. Нужно ли доплачивать? И через сколько придет подтверждение обмена?",
    "Привет) я вчера летел и оставил павербанк в кармашке кресла, рейс был из Сочи. как его найти? можно ли отправить почтой?",
    "Рейс задержали на 3 часа, объявили про технические неполадки. Положена ли мне какая-то компенсация? Если да, то как её оформить? Спасибо.",
    "Я опоздал на регистрацию на рейс S7-891 по уважительной причине. Можно ли оформить возврат средств за билет? Какие документы для этого нужны?",
    "Здрасте, а если я заболел и не полетел, могу ли я вернуть деньги за билет? У меня справка от врача есть. Сколько ждать возврат?",
    "Нужно срочно обменять билет на более раннюю дату. Как это сделать в приложении? Или только через поддержку? Тариф 'Промо'.",
    "После приземления рейса S7-246 я не дождался свой чемодан на ленте. Багаж потерялся! Что делать?! Куда звонить? Это ужасный сервис!",
    "Добрый день! Подскажите, пожалуйста, процедуру возврата билета в связи с отменой рейса. На какой срок затягивается процесс и на какую сумму я могу рассчитывать?",
    "Мой рейс перенесли, и новые даты меня не устраивают. Хочу обменять билет на другой рейс вашей компании. Возможен ли такой вариант?",
    "Здравствуйте, я летела рейсом S7-555 и оставила детскую игрушку в самолете. Для ребенка это очень важно. Помогите, пожалуйста, найти!",
    "Задержали рейс более чем на 4 часа, не предоставили даже питание. Какую компенсацию я могу требовать? И в какой срок её должны выплатить?",
    "По семейным обстоятельствам не смог улететь. Возможен ли возврат средств за невозвратный тариф? Где можно посмотреть условия?",
    "Хочу поменять дату вылета, так как изменились планы. Подскажите, какой штраф за обмен? И как долго обрабатывается заявка?",
    "После вылета рейса S7-777 обнаружила, что потеряла сережку, возможно, в зале ожидания. Есть ли у вас служба находок в аэропорту Домодедово?",
    "Рейс отменили в последний момент, пришлось ночевать в аэропорту. Требую компенсацию за моральный ущерб и дополнительные расходы! Как подать на компенсацию?",
    "Не смог лететь из-за болезни жены. Нужен возврат денег за два билета. Какие нужны документы кроме справки? Спасибо за помощь.",
    "Можно ли обменять билет с тарифом 'Базовый' на другой рейс? Если да, то какая разница в стоимости и комиссия? Жду ответа.",
    "Забыл в самолете планшет, рейс S7-321. Очень нужно найти! Куда можно позвонить по поводу утерянных вещей? Кто отвечает за это?",
    "Из-за задержки рейса я пропустил стыковочный рейс другой авиакомпании. Вы обязаны компенсировать мне убытки! Какой размер компенсации?",
    "Подскажите, как оформить возврат билета, если я ошибся с датой вылета? Деньги вернутся полностью или есть какие-то удержания?",
    "Планирую перенести поездку. Как произвести обмен билета на более позднюю дату? Есть ли ограничения по времени до вылета?",
    "Не могу найти свой рюкзак после выхода из самолета, рейс S7-999. В нем были важные документы! Помогите срочно!",
    "Задержка рейса составила 6 часов, причины не объявили. Какие мои права? Положена ли денежная компенсация или только ваучеры?",
    "Рейс отменили по вине авиакомпании. Хочу вернуть полную стоимость билета. Как правильно написать заявление на возврат?",
    "Нужно сдвинуть дату вылета на два дня вперед. Возможен ли обмен билета? Тариф 'Эконом'. Что для этого нужно сделать?",
    "Оставила зонт в кармашке кресла в самолете, рейс S7-456. Он очень дорогой как память. Можно ли его найти и как это сделать?",
    "Из-за задержки рейса мне пришлось снимать отель за свой счет. Авиакомпания обязана компенсировать эти расходы? Как оформить?",
    "Не смог полететь, так как закрыли границы. Как сделать возврат билета? Деньги вернутся на карту или в виде электронных средств?",
    "Хочу обменять билет на другой рейс той же авиакомпании. Какую комиссию я заплачу? И через сколько придет подтверждение?",
    "Потерял ключи от машины, возможно, в салоне самолета после рейса S7-135. Это критично для меня! Куда обращаться за помощью?"
]

for i, text in enumerate(test_texts_after):
    prediction = predict_level2_above_thresh(text, threshold=0.3)
    print(f"\nТекст {i+1}: {text}")
    print("Категории с вероятностью > 0.3:")
    for label, prob in prediction:
        print(f"  - {label}: {prob}")


Текст 1: Требую вернуть деньги за билет! Рейс был отменен, а альтернативный вариант мне не подошел. Как быстро происходит возврат средств на карту? Это просто невыносимо!
Категории с вероятностью > 0.3:
  - Возврат билета: 0.9876

Текст 2: Хочу перенести свой вылет на следующую неделю. Как это сделать? У меня тариф 'Гибкий'. Нужно ли доплачивать? И через сколько придет подтверждение обмена?
Категории с вероятностью > 0.3:
  - Обмен билета: 0.9718

Текст 3: Привет) я вчера летел и оставил павербанк в кармашке кресла, рейс был из Сочи. как его найти? можно ли отправить почтой?
Категории с вероятностью > 0.3:
  - Потерянная собственность: 0.9849

Текст 4: Рейс задержали на 3 часа, объявили про технические неполадки. Положена ли мне какая-то компенсация? Если да, то как её оформить? Спасибо.
Категории с вероятностью > 0.3:
  - Компенсация: 0.9816

Текст 5: Я опоздал на регистрацию на рейс S7-891 по уважительной причине. Можно ли оформить возврат средств за билет? Какие документы для этого

In [6]:
MODEL_PATH = "./drive/MyDrive/l2_models/На_борту"
MAX_LEN = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
mlb = joblib.load(f"{MODEL_PATH}/level2_mlb.joblib")
model.eval()

test_texts_after = [
  "Терморегуляция в салоне осуществлялась безупречно - стабильная температура, комфортная влажность, равномерное распределение воздуха. Но эргономические параметры кресел не соответствовали современным стандартам - неудобная посадка, недостаточная поддержка. Оперативность реакции обслуживающего персонала на запросы также требовала улучшения.",
  "Навыки межличностного общения стюардесс были исключительными - тактичность, внимательность, проактивность. Однако санитарно-гигиеническое состояние салона вызывало нарекания - видимая пыль, пятна на поверхностях. Техническая надежность развлекательной системы также оставляла желать лучшего - частые сбои в работе.",
  "Гастрономическая составляющая сервиса превосходила ожидания - разнообразие меню, свежесть продуктов, искусство подачи. Но функциональные возможности развлекательного комплекса были ограниченными - устаревшее программное обеспечение. Организация системы хранения ручной клади также требовала доработки - недостаток организованности.",
  "Соблюдение гигиенических протоколов было безупречным - регулярная уборка, дезинфекция, свежесть воздуха. Однако эргономические характеристики пассажирских мест не соответствовали требованиям комфорта - неудобная конструкция. Работа системы вентиляции создавала определенные неудобства из-за направления воздушных потоков.",
  "Пространственная организация пассажирского места была продумана оптимально - достаточное расстояние, регулируемые элементы, удобные аксессуары. Но поведенческий аспект других пассажиров создавал дискомфорт - повышенная активность. Качество предлагаемого развлекательного контента также не соответствовало заявленному уровню.",
  "Индивидуальный подход бортпроводников к пассажирам заслуживал высшей оценки - учет особых потребностей, персонализированное обслуживание. Однако параметры микроклимата в салоне оставляли желать лучшего - недостаточный воздухообмен. Конструктивные особенности багажных отделений также требовали усовершенствования - неудобства доступа.",
  "Искусство кулинарной подачи бортового питания было на высоте - эстетика, свежесть, разнообразие вкусов. Но техническая стабильность работы мультимедийной системы была неудовлетворительной - перебои в работе. Уровень внимания обслуживающего персонала к пассажирам также требовал повышения - редкое появление в салоне.",
  "Параметры акустического комфорта соответствовали premium стандартам - эффективная шумоизоляция, приятный звуковой фон. Однако эксплуатационные характеристики систем хранения багажа вызывали замечания - шум при работе. Соблюдение гигиенических норм в санитарных помещениях также не соответствовало общему уровню чистоты.",
  "Технологическое оснащение развлекательной системы отвечало современным требованиям - высокое качество, обширный контент, удобный интерфейс. Но стандарты бортового питания не соответствовали ожиданиям - температурный режим, ограниченность выбора. Организация процесса обслуживания также требовала оптимизации - временные промежутки.",
  "Функциональные характеристики багажных отделений были продуманы рационально - удобный доступ, достаточный объем, надежность. Однако профессиональные качества обслуживающего персонала оставляли вопросы - недостаточная инициатива. Качественные показатели воздушной среды в салоне также вызывали concerns - посторонние примеси.",
  "Система терморегуляции в салоне работала безупречно - оптимальные параметры, стабильность, равномерность распределения. Но эргономические решения пассажирских кресел не соответствовали современным стандартам комфорта - неудобная конфигурация. Скорость реакции обслуживающего персонала на запросы также требовала улучшения.",
  "Коммуникативная компетентность стюардесс заслуживала высшей оценки - культура общения, внимание к деталям, проактивная позиция. Однако санитарное состояние салона вызывало определенные concerns - видимые загрязнения. Надежность работы развлекательной системы также оставляла желать лучшего - технические неполадки.",
  "Гастрономические аспекты сервиса превосходили ожидания - разнообразие, свежесть, искусство presentation. Но функциональный потенциал развлекательного комплекса был ограниченным - устаревшее техническое оснащение. Организационные аспекты системы хранения ручной клади также требовали совершенствования - недостаток системности.",
  "Соблюдение гигиенических стандартов было безукоризненным - регулярность уборки, дезинфекция, свежесть. Однако эргономические параметры пассажирских мест не соответствовали требованиям комфорта - неудобная конструкция. Работа системы вентиляции создавала определенный дискомфорт из-за характеристик воздушных потоков.",
  "Организация пассажирского пространства была продумана рационально - достаточные габариты, регулируемость, удобство. Но поведенческий фактор со стороны других пассажиров создавал определенные неудобства - повышенная активность. Качественные характеристики развлекательного контента также не соответствовали заявленному уровню.",
  "Персонализированный подход бортпроводников к обслуживанию заслуживал высшей оценки - учет индивидуальных потребностей, внимательность. Однако параметры микроклимата в салоне оставляли желать лучшего - недостаточная вентиляция. Конструктивные решения багажных отделений также требовали усовершенствования - неудобства эксплуатации.",
  "Художественные аспекты подачи бортового питания были безупречны - эстетика, свежесть, разнообразие. Но техническая надежность мультимедийной системы была неудовлетворительной - системные сбои. Уровень внимания обслуживающего персонала к пассажирам также требовал повышения - редкое присутствие.",
  "Акустические параметры комфорта соответствовали premium стандартам - эффективная изоляция, благоприятный фон. Однако эксплуатационные характеристики систем хранения багажа вызывали замечания - шумность работы. Соблюдение гигиенических протоколов в санитарных зонах также не соответствовало общему уровню.",
  "Технологические возможности развлекательной системы отвечали современным требованиям - качество, контент, интерфейс. Но стандарты бортового питания не соответствовали ожиданиям - температурный режим, ассортимент. Организационные аспекты процесса обслуживания также требовали оптимизации - временные параметры.",
  "Функциональные характеристики багажных отделений были продуманы оптимально - доступность, вместимость, надежность. Однако профессиональные компетенции обслуживающего персонала оставляли вопросы - недостаточная активность. Качественные показатели воздушной среды в салоне также вызывали определенные concerns.",
  "Работа системы терморегуляции в салоне была безупречной - оптимальные параметры, стабильность, равномерность. Но эргономические решения пассажирских кресел не соответствовали современным стандартам комфорта - неудобства конструкции. Оперативность реакции обслуживающего персонала на запросы также требовала улучшения."
]

for i, text in enumerate(test_texts_after):
    prediction = predict_level2_above_thresh(text, threshold=0.3)
    print(f"\nТекст {i+1}: {text}")
    print("Категории с вероятностью > 0.3:")
    for label, prob in prediction:
        print(f"  - {label}: {prob}")


Текст 1: Терморегуляция в салоне осуществлялась безупречно - стабильная температура, комфортная влажность, равномерное распределение воздуха. Но эргономические параметры кресел не соответствовали современным стандартам - неудобная посадка, недостаточная поддержка. Оперативность реакции обслуживающего персонала на запросы также требовала улучшения.
Категории с вероятностью > 0.3:
  - Комфорт борт: 0.9725
  - Обслуживание борт: 0.9051

Текст 2: Навыки межличностного общения стюардесс были исключительными - тактичность, внимательность, проактивность. Однако санитарно-гигиеническое состояние салона вызывало нарекания - видимая пыль, пятна на поверхностях. Техническая надежность развлекательной системы также оставляла желать лучшего - частые сбои в работе.
Категории с вероятностью > 0.3:
  - Развлечения борт: 0.8392
  - Чистота борт: 0.8057
  - Обслуживание борт: 0.6999
  - Комфорт борт: 0.4341

Текст 3: Гастрономическая составляющая сервиса превосходила ожидания - разнообразие меню, свеже